# Baseline: PCA + k-nearest neighbours

PCA (100 components) on standardized train+validation data, then KNN for k = 1..14 on each of the six label sets. Accuracy is measured on the held-out test set.

In [1]:
import tarfile
from pathlib import Path
import pandas as pd

def find_data_dir(name="data", start=None):
    """Search the current folder and its parents for a folder called `name`."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / name
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"No '{name}' folder found above {start}")

DATA_DIR = find_data_dir()
print("DATA_DIR:", DATA_DIR)

# Find the archive (.gz, .tar.gz or .tgz) without hardcoding the exact name
src = next(DATA_DIR.glob("moanna_training_data*"), None)
if src is None:
    raise FileNotFoundError(f"No moanna_training_data* file in {DATA_DIR}")
print("Archive:", src)

out_dir = DATA_DIR / "extracted"
if not out_dir.exists():                      
    with tarfile.open(src, "r:*") as tar:     
        tar.extractall(out_dir)

# Locate the extracted files wherever they ended up
dat_path = next(out_dir.rglob("moanna_training_data.tsv"))
lab_path = next(out_dir.rglob("moanna_training_label.tsv"))


DATA_DIR: C:\Users\ASUS\Desktop\MOANNA-Autoencoder\data
Archive: C:\Users\ASUS\Desktop\MOANNA-Autoencoder\data\moanna_training_data.gz


In [2]:
dat_df_val_path = next(out_dir.rglob("moanna_validation_data.tsv"))
lab_df_val_path = next(out_dir.rglob("moanna_validation_label.tsv"))

dat_df_test_path = next(out_dir.rglob("moanna_testing_data.tsv"))
lab_df_test_path = next(out_dir.rglob("moanna_testing_label.tsv"))

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import preprocessing

In [4]:
dat_df = pd.read_csv(dat_path, sep="\t", index_col=0)
lab_df = pd.read_csv(lab_path, sep="\t", index_col=0)

dat_df_val = pd.read_csv(dat_df_val_path, sep="\t", index_col=0)
lab_df_val = pd.read_csv(lab_df_val_path, sep="\t", index_col=0)

dat_df_test = pd.read_csv(dat_df_test_path, sep="\t", index_col=0)
lab_df_test = pd.read_csv(lab_df_test_path, sep="\t", index_col=0)

In [5]:
label_df_concated = pd.concat([lab_df, lab_df_val])
label_df_concated

,ERStatus,HER2Status,Pam50Subtype,Pam50SubtypeSingleLuminal,BasalNonBasal,Luminal
MB-6098,1,1,0,0,0,0
MB-5420,0,0,0,0,0,0
MB-4763,1,0,1,1,1,0
MB-4873,0,1,3,2,1,2
MB-0639,1,1,0,0,0,0
...,...,...,...,...,...,...
MB-5313,0,1,1,1,1,0
MB-6080,0,1,3,2,1,2
MB-4708,0,1,2,2,1,1
MB-3031,0,0,1,1,1,0


In [6]:
data_df_concated = pd.concat([dat_df, dat_df_val])
data_df_concated.shape

(1689, 46776)

In [7]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [9]:
# Fit scaler and PCA on the training data only, then apply the same transform to the test set.
scaler = StandardScaler()
df_standardized = scaler.fit_transform(data_df_concated)
df_standardized_test = scaler.transform(dat_df_test)

pca = PCA(n_components=100)
data_df_with_PCA = pca.fit_transform(df_standardized)
data_df_with_PCA_test = pca.transform(df_standardized_test)

In [10]:
data_df_with_PCA.shape

(1689, 100)

In [11]:
data_df_with_PCA_test.shape

(631, 100)

In [12]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

label_names = ['ERStatus', 'HER2Status', 'Pam50Subtype',
               'Pam50SubtypeSingleLuminal', 'BasalNonBasal', 'Luminal']

# Build label arrays directly from the dataframes
list_all_label_train = [label_df_concated[name].values for name in label_names]
list_all_label_test = [lab_df_test[name].values for name in label_names]

# Run KNN for k = 1..14 on every label
res_list = []
for k in range(1, 15):
    res_list_for_each_k = []
    for i in range(len(label_names)):
        KNN_Obj = KNeighborsClassifier(n_neighbors=k).fit(data_df_with_PCA, list_all_label_train[i])
        Acc = metrics.accuracy_score(list_all_label_test[i], KNN_Obj.predict(data_df_with_PCA_test))
        res_list_for_each_k.append(Acc)
    res_list.append(res_list_for_each_k)

# Build the results table: rows = k, columns = labels
results_df = pd.DataFrame(res_list,
                          columns=label_names,
                          index=range(1, len(res_list) + 1))
results_df.index.name = 'k'
results_df['Average'] = results_df.mean(axis=1)

# Show as percentages, highlight the best value in each column
styled = (results_df * 100).style \
    .format("{:.2f}") \
    .highlight_max(axis=0, color='lightgreen') \
    .set_caption("KNN accuracy (%) for each k and label")
display(styled)

# Short summary
best_k = results_df['Average'].idxmax()
print(f"\nBest k by average accuracy: k = {best_k} "
      f"({results_df.loc[best_k, 'Average'] * 100:.2f}%)")
print("\nBest k for each label:")
for name in label_names:
    k_best = results_df[name].idxmax()
    print(f"  {name:<28} k = {k_best:<3} acc = {results_df.loc[k_best, name] * 100:.2f}%")

,ERStatus,HER2Status,Pam50Subtype,Pam50SubtypeSingleLuminal,BasalNonBasal,Luminal,Average
k,,,,,,,
1,90.65,78.92,69.89,87.16,96.35,72.27,82.54
2,90.17,77.81,72.42,86.69,95.72,75.12,82.99
3,91.76,80.51,72.11,90.65,97.31,74.48,84.47
4,90.81,80.67,75.12,91.92,97.62,77.02,85.53
5,91.92,80.19,75.12,91.60,98.26,76.55,85.60
6,90.33,80.67,75.75,91.44,98.57,77.97,85.79
7,91.60,79.40,77.02,91.60,98.57,78.45,86.11
8,91.28,79.71,77.34,92.08,98.42,78.45,86.21
9,91.28,79.08,77.97,91.44,97.94,78.92,86.11



Best k by average accuracy: k = 12 (86.27%)

Best k for each label:
  ERStatus                     k = 5   acc = 91.92%
  HER2Status                   k = 4   acc = 80.67%
  Pam50Subtype                 k = 11  acc = 78.29%
  Pam50SubtypeSingleLuminal    k = 8   acc = 92.08%
  BasalNonBasal                k = 6   acc = 98.57%
  Luminal                      k = 11  acc = 79.56%
